Catastrophic cancellation. Compute the variance of [1000000.0, 1000001.0, 1000002.0] using the naive formula E[x^2] - E[x]^2 in float32. Then compute it using Welford's online algorithm. Compare the errors against the true variance (0.6667).

In [1]:
import numpy as np
data = [1000000.0, 1000001.0, 1000002.0]
n=len(data)
true_variance=np.var(np.array(data,dtype=np.float64))
print(f"True variance (float64): {true_variance:.10f}")
print(f"Expected: 0.6667")
print()

True variance (float64): 0.6666666667
Expected: 0.6667



- Compute E[x] and E[x^2]


In [2]:
data_f32 = np.array(data, dtype=np.float32)
mean_f32 = np.mean(data_f32)
mean_sq_f32 = np.mean(data_f32**2)
naive_variance = mean_sq_f32 - mean_f32**2
print("=== Naive Formula (float32) ===")
print(f"E[x]   = {mean_f32}")
print(f"E[x^2] = {mean_sq_f32}")
print(f"E[x]^2 = {mean_f32**2}")
print(f"Variance = E[x^2] - E[x]^2 = {naive_variance}")
print(f"Error: {abs(naive_variance - true_variance):.6f}")
print(f"Relative error: {abs(naive_variance - true_variance)/true_variance * 100:.2f}%")
print()

=== Naive Formula (float32) ===
E[x]   = 1000001.0
E[x^2] = 1000001961984.0
E[x]^2 = 1000002027520.0
Variance = E[x^2] - E[x]^2 = -65536.0
Error: 65536.666667
Relative error: 9830500.00%



- Method 2: Welford's online algorithm in float32

In [3]:
def welford_variance(data):
    n = 0
    mean = np.float32(0.0)
    M2 = np.float32(0.0)
    
    for x in data:
        x = np.float32(x)
        n += 1
        delta = x - mean
        mean += delta / np.float32(n)
        delta2 = x - mean
        M2 += delta * delta2
    
    if n < 2:
        return np.float32(0.0)
    else:
        return M2 / np.float32(n)

In [4]:
welford_var = welford_variance(data)

print("=== Welford's Online Algorithm (float32) ===")
print(f"Variance = {welford_var}")
print(f"Error: {abs(welford_var - true_variance):.6f}")
print(f"Relative error: {abs(welford_var - true_variance)/true_variance * 100:.6f}%")
print()

=== Welford's Online Algorithm (float32) ===
Variance = 0.6666666865348816
Error: 0.000000
Relative error: 0.000003%



In [5]:
# Compare catastrophic cancellation
print("=== Analysis ===")
print(f"Naive formula suffers from catastrophic cancellation:")
print(f"  E[x^2] = {mean_sq_f32} (~1e12)")
print(f"  E[x]^2 = {mean_f32**2} (~1e12)")
print(f"  Difference = {naive_variance} (should be ~0.667)")
print(f"  We're subtracting two large nearly-equal numbers,")
print(f"  losing ~6-7 significant digits in float32 (which has ~7 digits).")
print()
print(f"Welford's algorithm avoids this by working with deviations")
print(f"from the running mean, keeping numbers small and well-conditioned.")

=== Analysis ===
Naive formula suffers from catastrophic cancellation:
  E[x^2] = 1000001961984.0 (~1e12)
  E[x]^2 = 1000002027520.0 (~1e12)
  Difference = -65536.0 (should be ~0.667)
  We're subtracting two large nearly-equal numbers,
  losing ~6-7 significant digits in float32 (which has ~7 digits).

Welford's algorithm avoids this by working with deviations
from the running mean, keeping numbers small and well-conditioned.
